## Loan Underwriting Agent Evaluation

In [2]:
import os, json
from langchain_mcp_adapters.client import MultiServerMCPClient

SERVER_PATH = os.path.abspath("wealthpilot_mcp_server.py")

# ---------------------------------------------------------------------------------------------

import sys, io
import mcp.client.stdio as _mcp_stdio
import langchain_mcp_adapters.sessions as _mcp_sessions

# 1. Open the log once, not once per cell re-run (the old version leaked a file descriptor each time).
if "MCP_SERVER_LOG" not in globals() or MCP_SERVER_LOG.closed:
    MCP_SERVER_LOG = open("mcp_server.log", "a", encoding="utf-8")

# 2. Recover the true original. A patched function carries a marker, so we can unwrap it
#    instead of trusting whatever is currently bound to the module attribute.
_current = _mcp_stdio.stdio_client
_true_original = getattr(_current, "__pre_patch__", None) or _current
if getattr(_true_original, "__module__", "") == "__main__":
    raise RuntimeError("stdio_client is already patched by an unmarked patch — restart the kernel.")
_mcp_stdio._pre_patch_stdio_client = _true_original

# 3. Bind `original` and `log` as keyword-only DEFAULTS -> evaluated once, at def time.
#    They can never be re-resolved through the notebook's global namespace.
def _stdio_client_logging_to_file(server, errlog=None, *, _orig=_true_original, _log=MCP_SERVER_LOG):
    return _orig(server, errlog if errlog is not None else _log)

_stdio_client_logging_to_file.__pre_patch__ = _true_original   # makes re-runs safe forever

_mcp_stdio.stdio_client = _stdio_client_logging_to_file
_mcp_sessions.stdio_client = _stdio_client_logging_to_file

In [3]:
mcp_client = MultiServerMCPClient({
    "project": {
        "transport": "stdio",     
        "command": sys.executable, 
        "args": [SERVER_PATH],
    }
})

mcp_tools = await mcp_client.get_tools()
TOOLS_BY_NAME = {t.name: t for t in mcp_tools}

print("DISCOVERED TOOLS")
print("-------------------------------------------------------")
for t in mcp_tools:
    print(f"  {t.name}{tuple(t.args)}")
    print(f"      {t.description.splitlines()[0]}")

DISCOVERED TOOLS
-------------------------------------------------------
  applicant_details('applicant_id',)
      Retrieve applicant financial and operating fields for an applicant ID.
  loan_application_details('applicant_id',)
      Retrieve loan and collateral fields for an applicant ID.
  bureau_reports('applicant_id',)
      Retrieve bureau score and adverse-credit fields for an applicant ID.
  risk_grades('dscr', 'bureau_score')
      Retrieve the risk grade for a DSCR and bureau score.
  sector_policies('sector_name',)
      Retrieve the policy floor and tenor cap for a sector.
  query_knowledge_base('query', 'k', 'source')
      Fetch information from the knowledge base which is based on the company internal documents


In [6]:
import time
from langchain_litellm import ChatLiteLLM
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
from pydantic import BaseModel, Field

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
LLM_MODEL = os.getenv("LLM_MODEL", "gemini/gemini-flash-lite-latest")
chat_model = ChatLiteLLM(model=LLM_MODEL, temperature=0, api_key=os.getenv("GEMINI_API_KEY", GEMINI_API_KEY))

In [7]:
LLM_CALL_PAUSE_SECONDS = 3.0

def _probe_model() -> bool:
    """One real, tiny call. A present-but-invalid key fails HERE, once, with a clear message -
    instead of failing inside a graph node twenty cells later where it looks like a LangGraph bug."""
    try:
        chat_model.invoke([HumanMessage(content="Reply with the single word: ok")])
        return True
    except Exception as e:
        print(f"  model unavailable ({type(e).__name__}: {str(e)[:110]})")
        return False

LLM_ENABLED = _probe_model()

def ask(system: str, user: str) -> str | None:
    """Plain text -> text. Returns None (never raises) so callers can fall back deterministically."""
    if not LLM_ENABLED:
        return None
    try:
        msgs = [SystemMessage(content=system), HumanMessage(content=user)]
        return chat_model.invoke(msgs).content.strip()
    except Exception as e:
        print(f"  [model call failed: {type(e).__name__}] falling back")
        return None
    finally:
        time.sleep(LLM_CALL_PAUSE_SECONDS)  # give the rate limit room to breathe before the next call


_STRUCTURED_CACHE: dict[type, object] = {}

def ask_structured(system: str, user: str, schema: type[BaseModel]) -> BaseModel | None:
    """Text -> a VALIDATED Pydantic object. Returns None on any failure. """
    if not LLM_ENABLED:
        return None
    try:
        runnable = _STRUCTURED_CACHE.setdefault(schema, chat_model.with_structured_output(schema))
        return runnable.invoke([SystemMessage(content=system), HumanMessage(content=user)])
    except Exception as e:
        print(f"  [structured call failed: {type(e).__name__}] falling back")
        return None
    finally:
        time.sleep(LLM_CALL_PAUSE_SECONDS)  # give the rate limit room to breathe before the next call

print("LLM_ENABLED :", LLM_ENABLED, "|", LLM_MODEL if LLM_ENABLED else "running on deterministic fallbacks")

LLM_ENABLED : True | gemini/gemini-flash-lite-latest


In [8]:
def scoped(role: str):
    """Enforce write permissions. A violation raises instead of silently corrupting state.

    Handles async nodes too - in Lab B the researcher becomes async (it awaits an MCP call), and
    a decorator that only understood sync functions would wrap a coroutine and fail obscurely."""
    allowed = AGENT_SCOPES[role] | {"log"}

    def enforce(update):
        illegal = set(update) - allowed
        if illegal:
            raise PermissionError(
                f"agent '{role}' wrote outside its scope: {sorted(illegal)}; allowed={sorted(allowed)}")
        return update

    def decorate(fn):
        if inspect.iscoroutinefunction(fn):
            async def awrapper(state):
                return enforce(await fn(state))
            awrapper.__name__ = fn.__name__
            return awrapper
        def wrapper(state):
            return enforce(fn(state))
        wrapper.__name__ = fn.__name__
        return wrapper
    return decorate

In [9]:
# Team state, agent scopes, and the enforcement decorator.
from typing import TypedDict, Annotated, Literal, List
from operator import add
import inspect, re, json
from langgraph.graph import StateGraph, START, END

MAX_REVISIONS = 2          # the loop budget. Everything about convergence hangs off this number.

class UnderWritingState(TypedDict):
    raw: str
    proposed_annual_debt_service_inr: float
    dscr: float
    sector_dscr_floor: float
    meets_dscr_floor: bool
    leverage_ratio: float
    current_ratio: float
    revenue_variance_pct: float
    internal_grade: Literal["A1", "A2", "B1", "B2", "C1", "C2", "D1", "D2"]
    indicative_rate_pct: float
    reason_codes: str
    risk_band: Literal["approve", "borderline", "decline"]
    approval_authority: Literal["Credit Officer", "Board Credit Committee", "Credit Committee"]
    auto_approve_allowed: bool
    requires_human_signoff: bool
    loan_application: dict
    missing_fields: List[str]
    log: Annotated[list[str], add]
    revision_count: int
    critic_decision: Literal[None, "pass", "review"]
    critic_feedback: str
    final_status: str


# write scope
AGENT_SCOPES = {
    "analyst": {"loan_application", "revision_count", "critic_decision"},
    "risk_reviewer": {
        "proposed_annual_debt_service_inr",
        "dscr", "sector_dscr_floor", "meets_dscr_floor", "leverage_ratio",
        "current_ratio", "revenue_variance_pct", "internal_grade", "indicative_rate_pct",
        "reason_codes", "risk_band"
        },
    "compliance_reviewer": {"approval_authority", "auto_approve_allowed", "requires_human_signoff", 'final_status'},
    "critic": {"critic_decision", "critic_feedback", "final_status"},
}

## Systems Prompts for Agents
SYSTEM_PROMPTS = {
 "analyst": "You read the raw loan application and extract critical values required for underwriting. " 
            "If they are missing, set them to an empty string. Do not set the missing fields to 0", 
 "risk_reviewer": "You receive the loan application in JSON format. If there are missing fields, "
                    " use the available tools to fetch the values for those missing fields.",
 "critic": "You review the generated underwriting report. Refer to the knowledge base, fetch the documents "
             "and verify that the risk grade assigned to the loan application is accurate. "
}

# read scope
def context_for(role: str, state: UnderWritingState) -> dict:
    """READ scoping: hand each agent only the slice it needs. """

    if role == "analyst":      return {"raw": state["raw"], "missing_fields": state["missing_fields"]}
    if role=='risk_reviewer':  return state['loan_application']
    if role=='compliance_reviewer': return {'risk_band': state['risk_band'],
                                            'requested_amount': state["loan_application"]['requested_amount_inr'],
                                            "internal_grade": state['internal_grade']
                                            }
    if role == "critic":   return {"dscr": state["dscr"],
                                    "bureau_score": state["loan_application"]['bureau_score'],
                                       "internal_grade": state['internal_grade']
                                       }
    return {}


# The typed contracts each agent must return

class Analyst(BaseModel):
    """The analyst's output."""
    applicant_id: str = Field(description="Unique identifier for the applicant. It should be ASH-A-XXXXX where X is a digit")
    application_id: str  = Field(description="Unique identifier for the applicantion. It should be ASH-L-XXXXX where X is a digit")
    sector: str = Field(description="Industry sector of the applicant")
    requested_amount_inr: float | None = Field(description="Requested loan amount in INR")
    requested_tenor_months: int | None = Field(description="Requested loan tenor in months")
    bureau_score: int | None = Field(description="Credit bureau score")
    ebitda_inr: int | None = Field(description="EBITDA in INR")
    existing_annual_debt_service_inr: int | None = Field(description="Existing annual debt service in INR")
    months_operating: int | None = Field(description="Number of months operating")
    annual_revenue_inr: int | None = Field(description="Verified Annual Revenue in INR")
    existing_debt_inr: int | None = Field(description="Existing debt in INR")
    current_assets_inr: int | None = Field(description="Current assets in INR")
    current_liabilities_inr: int | None = Field(description="Current liabilities in INR")
    collateral_value_inr: int | None = Field(description="Collateral value in INR")

class Critic(BaseModel):
    """The critic's output"""
    critic_decision: Literal["pass", "review"] = Field(description="Reply with a single word 'Pass' if all the tests are passed else 'review'")
    critic_feedback: str = Field(description="one short sentence explaining the verdict")

In [10]:
@scoped("analyst")
def analyst_node(state: UnderWritingState) -> dict:
    ctx = context_for("analyst", state)
    is_revision = state["critic_decision"] == 'review'

    CORRECTION_PROMPT = ""
    if is_revision:
        CORRECTION_PROMPT = f"And fix these issues {state['critic_feedback']}"

    out = ask_structured(
        SYSTEM_PROMPTS["analyst"] + CORRECTION_PROMPT,
        f"Raw Application:\n{ctx['raw']}",
        Analyst)

    if out is None: # Fallback
        match_applicant = re.search(r'ASH-A-\d+', ctx['raw'])
        match_application = re.search(r'ASH-L-\d+', ctx['raw'])
        if match_applicant and match_application:
            applicant_id = match_applicant.group(0)
            application_id = match_application.group(0)
            loan_app_dict = {'applicant_id': applicant_id,
                                'application_id' : application_id,
                            'sector': '',
                            'requested_amount_inr': None,
                            'requested_tenor_months': None,
                            'bureau_score': None,
                            'ebitda_inr': None,
                            'existing_annual_debt_service_inr': None,
                            'months_operating': None,
                            'annual_revenue_inr': None,
                            'existing_debt_inr': None,
                            'current_assets_inr': None,
                            'current_liabilities_inr': None,
                            'collateral_value_inr': None,
                            'declared_annual_revenue_inr': '',
                            'active_trade_lines': '',
                            'dpd_90_ever': '',
                            'written_off_amount_inr': '',
                            'dscr_floor': '',
                            'sector_max_tenor_months': '',
                            'at_sector_cap': ''
                            }
    else:
        loan_app_dict = out.model_dump()
        loan_app_dict.update(
            {
                "declared_annual_revenue_inr": "",
                "active_trade_lines": "",
                "dpd_90_ever": "",
                "written_off_amount_inr": "",
                "dscr_floor": "",
                "sector_max_tenor_months": "",
                "at_sector_cap": ""
            }
        )
    return {'loan_application' : loan_app_dict,
            "revision_count": state["revision_count"] + (1 if is_revision else 0),
            "log": ["Extracted data from the raw intake record"],
            "critic_decision": ""
            }

# Validation node - Pure python function
def validate_node(state: UnderWritingState) -> dict:
    missing = []
    for field in state["loan_application"]:
        if state["loan_application"][field] in [None, ""]:
            missing.append(field)
    return {"missing_fields": missing, "log": [f"Validator: {len(missing)} fields are missing"]}

# Route node - Pure python function
def route_after_validation(state: UnderWritingState) -> str:
    if state["missing_fields"]: return "fetch"
    else: return "risk_review"

def parse_tool_result(raw):
    """MCP tool results arrive as CONTENT BLOCKS, not plain Python values."""
    if isinstance(raw, list) and raw and isinstance(raw[0], dict) and "text" in raw[0]:
        raw = "".join(b.get("text", "") for b in raw if b.get("type") == "text")
    if isinstance(raw, str):
        try:
            return json.loads(raw)
        except json.JSONDecodeError:
            return raw
    return raw

# Tool calling node - uses LLM
async def tool_call_and_execute_node(state: UnderWritingState) -> dict:
    """The policy node: look at the missing fields so far, decide to call a tool to fetch the field value."""

    current_dict = state['loan_application']
    log = []

    if LLM_ENABLED:
        messages = [SystemMessage(content="You fill the missing fields. Look at all the missing fields and use the tool available to you"
                          "to fetch the value for those fields."),
                          HumanMessage(content=f"Applicant ID is {current_dict['applicant_id']}, sector is {current_dict['sector']}. Missing fields are: {state['missing_fields']}")]
        model = chat_model.bind_tools(mcp_tools)
        response = await model.ainvoke(messages)
        if response.tool_calls:
            for c in response.tool_calls:
                tool_call_result = await TOOLS_BY_NAME[c["name"]].ainvoke(c["args"])
                current_dict.update(parse_tool_result(tool_call_result[0]['text']))
                log.append(f"Tool call: {c['name']} with arg {c['args']}.")  

    else: #fallback
        tool_call_result = await TOOLS_BY_NAME["applicant_details"].ainvoke({'applicant_id': current_dict['applicant_id']})
        current_dict.update(parse_tool_result(tool_call_result[0]['text']))
        log.append(f"Tool call: applicant_details with arg 'applicant_id': {current_dict['applicant_id']}.")

        tool_call_result = await TOOLS_BY_NAME["loan_application_details"].ainvoke({'applicant_id': current_dict['applicant_id']})
        current_dict.update(parse_tool_result(tool_call_result[0]['text']))
        log.append(f"Tool call: loan_application_details with arg 'applicant_id': {current_dict['applicant_id']}.")

        tool_call_result = await TOOLS_BY_NAME["bureau_reports"].ainvoke({'applicant_id': current_dict['applicant_id']})
        current_dict.update(parse_tool_result(tool_call_result[0]['text']))
        log.append(f"Tool call: bureau_reports with arg 'applicant_id': {current_dict['applicant_id']}.")

        tool_call_result = await TOOLS_BY_NAME["sector_policies"].ainvoke({'sector_name': current_dict['sector']})
        current_dict.update(parse_tool_result(tool_call_result[0]['text']))
        log.append(f"Tool call: bureau_reports with arg 'applicant_id': {current_dict['sector']}.")

    return {'loan_application': current_dict, "missing_fields": [], 'log': log}


# Python function with a single tool call

@scoped("risk_reviewer")
async def risk_reviewer_node(state: UnderWritingState)-> dict:
    ctx = context_for("risk_reviewer", state)
    codes=[]

    r = 14.0 / 1200
    emi = ctx['requested_amount_inr'] * r / (1 - (1 + r) ** -ctx['requested_tenor_months'])
    proposed =  round(emi * min(12, ctx['requested_tenor_months']))
        
    if ctx['existing_annual_debt_service_inr'] is not None and proposed is not None:
        dscr = round(ctx['ebitda_inr'] / (ctx['existing_annual_debt_service_inr'] + proposed),2)
    
    tool_call_result = await TOOLS_BY_NAME["risk_grades"].ainvoke({'dscr': dscr, 'bureau_score': ctx['bureau_score']})
    risk_grade_dict = parse_tool_result(tool_call_result[0]['text'])

    if ctx['ebitda_inr'] and ctx['existing_debt_inr'] is not None:
        leverage = round((ctx['existing_debt_inr'] + ctx['requested_amount_inr']) / ctx['ebitda_inr'], 2)

    if ctx['current_assets_inr'] is not None and ctx['current_liabilities_inr']:
        current_ratio = round(ctx['current_assets_inr'] / ctx['current_liabilities_inr'], 2)

    if ctx["declared_annual_revenue_inr"] is not None and ctx["annual_revenue_inr"]:
        variance = round(100 * (ctx["declared_annual_revenue_inr"] - ctx["annual_revenue_inr"]) / ctx["annual_revenue_inr"], 1)
    
    # adding reason codes
    if dscr is not None and dscr < ctx['dscr_floor']:
        codes.append("DSCR_BELOW_FLOOR")

    thin = (ctx['months_operating'] is not None and ctx['months_operating'] < 24) or (
        ctx['active_trade_lines'] is not None and ctx['active_trade_lines'] < 1
    )
    if thin:
        codes.append("THIN_FILE")
    adverse = ctx['bureau_score'] is not None and ctx['bureau_score'] < 450
    if adverse:
        codes.append("ADVERSE_BUREAU")
    if ctx['dpd_90_ever'] or ctx['written_off_amount_inr'] >= 500_000:
        codes.append("PAST_DELINQUENCY")
    if variance is not None and abs(variance) > 10:
        codes.append("DOC_MISMATCH")
    if variance is not None and abs(variance) > 25:
        codes.append("FRAUD_HOLD")
    if ctx['at_sector_cap']==1:
        codes.append("SECTOR_CAP")
    if leverage is not None and leverage > 6.0:
        codes.append("LEVERAGE_ABOVE_LIMIT")
    if current_ratio is not None and current_ratio < 1.10:
        codes.append("LIQUIDITY_BELOW_FLOOR")

    too_young = ctx['months_operating'] is not None and ctx['months_operating'] < 24
    decline = (dscr is not None and dscr < 1.0) or adverse or too_young or risk_grade_dict['grade'] == "D2"
    
    if not decline and risk_grade_dict['grade'] is not None:
        grade_tenor = risk_grade_dict['max_tenor_months']
        cover = risk_grade_dict['collateral_cover_pct']

        if ctx['requested_tenor_months'] > min(grade_tenor, ctx['sector_max_tenor_months']):
            codes.append("TENOR_ABOVE_LIMIT")
        if ctx['collateral_value_inr'] is not None and ctx['collateral_value_inr'] < ctx['requested_amount_inr'] * cover / 100:
            codes.append("COLLATERAL_SHORTFALL")
        if risk_grade_dict['grade'] in ("C2", "D1"):
            codes.append("GRADE_REQUIRES_REVIEW")

    incomplete = (
        dscr is None or leverage is None or current_ratio is None
        or ctx['bureau_score'] is None or ctx['months_operating'] is None
    )
    if incomplete:
        codes.append("INCOMPLETE_APPLICATION")

    if decline:
        band = "decline"
    elif codes or incomplete:
        band = "borderline"
    else:
        band = "approve"

    return {
        'proposed_annual_debt_service_inr': proposed,
        'dscr' : dscr,
        'sector_dscr_floor': ctx['dscr_floor'],
        'meets_dscr_floor' : dscr >= ctx['dscr_floor'],
        'leverage_ratio' : leverage,
        'current_ratio' : current_ratio,
        'revenue_variance_pct' : variance,
        'internal_grade': risk_grade_dict['grade'],
        'indicative_rate_pct': risk_grade_dict['indicative_rate_pct'],
        'reason_codes' : ";".join(codes) or "APPROVED_STANDARD",
        "risk_band": band,
        'log': ["risk reviewer: calculated critical metrics and the risk band"]
    }

@scoped("compliance_reviewer")
def compliance_reviewer_node(state: UnderWritingState)-> dict:
    ctx = context_for("compliance_reviewer", state)
    
    auto = (
    ctx['risk_band'] == "approve"
    and ctx['requested_amount'] <= 1_000_000
    and ctx['internal_grade'] in ("A1", "A2", "B1", "B2", "C1") 
)
    _AUTHORITIES = [
    ("Credit Officer", 1_000_000),
    ("Credit Committee", 5_000_000),
    ("Board Credit Committee", None),
    ]

    return {
        "approval_authority": next(name for name, limit in _AUTHORITIES if limit is None or ctx['requested_amount']  <= limit),
        "auto_approve_allowed": auto,
        "requires_human_signoff": not auto,
        "final_status" : "Compliance Review passed. Underwriting report is ready!",
        "log": ["Compliance Review done"]
    }

## LLM as a Judge - independently verifies the assigned risk grade against the company's internal documents
@scoped("critic")
async def critic_node(state: UnderWritingState)-> dict:
    ctx = context_for("critic", state)

    if LLM_ENABLED:
        query = "What is the Risk Grading matrix that the underwriting system should follow?"

        #Check - manual invoking a tool
        tool_call_result = await TOOLS_BY_NAME["query_knowledge_base"].ainvoke({"query": query, "k": 1, "source": "risk-grading"})
        kb_context = parse_tool_result(tool_call_result[0]['text'])

        out = ask_structured(
        SYSTEM_PROMPTS["critic"],
        f"Given the context:\n{kb_context}.\n\n"
        f"Verify that BOTH the criteria are met: {ctx['dscr']} >= the minimum DSCR required for {ctx['internal_grade']}"
        f"AND {ctx['bureau_score']} >=  the minimum bureau score required for {ctx['internal_grade']} ",
        Critic)
        critic_output = out.model_dump()
    else: # Fallback
        _GRADES = [
            ("A1", 2.00, 750, 150, 84, 0),
            ("A2", 1.75, 700, 200, 72, 25),
            ("B1", 1.50, 650, 275, 60, 50),
            ("B2", 1.35, 600, 350, 60, 75),
            ("C1", 1.25, 550, 450, 48, 100),
            ("C2", 1.15, 500, 575, 36, 125),
            ("D1", 1.00, 450, 725, 24, 150),
            ("D2", 0.00, 0, 900, 12, 150)
            ]

        expected_grade =  next(g for g, d, b, *_ in _GRADES if ctx['dscr'] >= d and ctx['bureau_score'] >= b)
        result = 'Pass' if expected_grade == ctx['internal_grade'] else 'review'
        critic_output = {'critic_decision': result, 'critic_feedback': "Fallback"}

    if critic_output['critic_decision']=='review':
        critic_output['final_status'] = 'Revision required'
    return critic_output

# Route node - Pure python function
def route_after_critic(state: UnderWritingState) -> str:
    if state["critic_decision"]=='review' and state["revision_count"] < MAX_REVISIONS: return "revise"
    else: return "end"

In [ ]:
#await critic_node({'dscr': 3.65, 'internal_grade': 'C1', 'loan_application': {'bureau_score': 594}})

{'critic_decision': 'Pass', 'critic_feedback': 'Fallback'}

In [11]:
g = StateGraph(UnderWritingState)
g.add_node("analyse", analyst_node)
g.add_node("validate", validate_node)
g.add_node("tool_call", tool_call_and_execute_node)
g.add_node("risk_review", risk_reviewer_node)
g.add_node("compliance_review", compliance_reviewer_node)
g.add_node("critic", critic_node)
#g.add_node("finalize", finalize_node)

g.add_edge(START, "analyse")
g.add_edge("analyse", "validate")
g.add_conditional_edges("validate", route_after_validation,
                        {"fetch": "tool_call", "risk_review": "risk_review"})
g.add_edge("tool_call", "validate")
g.add_edge("risk_review", "compliance_review")
g.add_edge("compliance_review", "critic")
g.add_conditional_edges("critic", route_after_critic,
                        {"revise": "analyse", "end": END})

multi_agent_graph= g.compile()

## Load the Input dataset

In [12]:
## read the jsonl file - the loan application records

with open("../data/wealthpilot/intake/records.jsonl") as f:
    input_records = [json.loads(line) for line in f]
print(f"Loaded {len(input_records)} records from records.jsonl.")

Loaded 200 records from records.jsonl.


In [13]:
def get_input_record(app_id):
    "Get record by application_id from the loaded records. Returns None if not found."
    return next((record for record in input_records if record['application_id'] == app_id), None)

## Create Golden Dataset

In [14]:
import pandas as pd

df = pd.read_csv('../data/wealthpilot/mock_api/underwriting_reference.csv')
ref_underwriting = df.to_dict('records')

ref_underwriting[0]

{'application_id': 'ASH-L-00000',
 'proposed_annual_debt_service_inr': 369118,
 'dscr': 1.93,
 'sector_dscr_floor': 1.35,
 'meets_dscr_floor': True,
 'leverage_ratio': 1.78,
 'current_ratio': 1.77,
 'revenue_variance_pct': 8.3,
 'internal_grade': 'D2',
 'indicative_rate_pct': 18.5,
 'reason_codes': 'THIN_FILE;ADVERSE_BUREAU;PAST_DELINQUENCY',
 'risk_band': 'decline',
 'approval_authority': 'Credit Officer',
 'auto_approve_allowed': False,
 'requires_human_signoff': True}

In [15]:
# select 10 records that are present in both the intake records and the reference underwriting dataset

GOLDEN_DATASET = []
i = 0
while len(GOLDEN_DATASET) < 10 and i < len(ref_underwriting):
    app_id = ref_underwriting[i]['application_id']
    input_record = get_input_record(app_id)
    if input_record:
        result = {"input": input_record, "expected_outcome": dict(list(ref_underwriting[i].items())[1:])}
        GOLDEN_DATASET.append(result)
    i += 1

In [16]:
len(GOLDEN_DATASET)

10

In [17]:
# compare two dictionaries and give score of 1 if they are equal else 0
def compare_dicts(dict1, dict2):
    return 1 if dict1 == dict2 else 0

In [103]:
final_reports = []
scores = []

for record in GOLDEN_DATASET:
    RAW_INTAKE = str(record['input'])
    
    seed_report = {
    "raw": RAW_INTAKE,
    "proposed_annual_debt_service_inr": "",
    "dscr": "",
    "sector_dscr_floor": "",
    "meets_dscr_floor": "",
    "leverage_ratio": "",
    "current_ratio": "",
    "revenue_variance_pct": "",
    "internal_grade": "",
    "indicative_rate_pct": "",
    "reason_codes": "",
    "risk_band": "",
    "approval_authority": "",
    "auto_approve_allowed": False,
    "requires_human_signoff": True,
    "loan_application": {},
    "missing_fields": [],
    "log": [],
    "revision_count": 0,
    "critic_decision": "",
    "critic_feedback": "",
    "final_status": ""
    }

    # Invoke the workflow
    dcfg = {"configurable": {"thread_id": "committe-run"}}
    report = await multi_agent_graph.ainvoke(seed_report, dcfg)

    keys_to_extract = [
    'proposed_annual_debt_service_inr',
    'dscr',
    'sector_dscr_floor',
    'meets_dscr_floor',
    'leverage_ratio',
    'current_ratio',
    'revenue_variance_pct',
    'internal_grade',
    'indicative_rate_pct',
    'reason_codes',
    'risk_band',
    'approval_authority',
    'auto_approve_allowed',
    'requires_human_signoff'
    ]

    # Dictionary comprehension
    final_report = {key: report[key] for key in keys_to_extract}
    final_reports.append(final_report)

    score = compare_dicts(final_report, record['expected_outcome'])
    if score == 1:
        print(f"{record['input']['application_id']}: Underwriting report matches exactly with the ground truth")
    scores.append(score)
    print("-" * 40)

Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x000001C9DE828FD0>


ASH-L-00003: Underwriting report matches exactly with the ground truth
----------------------------------------
ASH-L-00005: Underwriting report matches exactly with the ground truth
----------------------------------------
ASH-L-00006: Underwriting report matches exactly with the ground truth
----------------------------------------
ASH-L-00008: Underwriting report matches exactly with the ground truth
----------------------------------------
ASH-L-00009: Underwriting report matches exactly with the ground truth
----------------------------------------
ASH-L-00012: Underwriting report matches exactly with the ground truth
----------------------------------------
ASH-L-00016: Underwriting report matches exactly with the ground truth
----------------------------------------
ASH-L-00019: Underwriting report matches exactly with the ground truth
----------------------------------------
ASH-L-00020: Underwriting report matches exactly with the ground truth
---------------------------------

In [106]:
if sum(scores) == 10:
    print("All records matched successfully")

All records matched successfully


## Package as a FastAPI service

In [18]:
import os, json, re, time, random, asyncio, threading, inspect

LLM_ENABLED  = False

keys_to_extract = [
'proposed_annual_debt_service_inr',
'dscr',
'sector_dscr_floor',
'meets_dscr_floor',
'leverage_ratio',
'current_ratio',
'revenue_variance_pct',
'internal_grade',
'indicative_rate_pct',
'reason_codes',
'risk_band',
'approval_authority',
'auto_approve_allowed',
'requires_human_signoff'
]

In [32]:
from fastapi import FastAPI, HTTPException

class ChatRequest(BaseModel):
    applicant_id: str
    application_id: str

class ChatResponse(BaseModel):
    proposed_annual_debt_service_inr: float
    dscr: float
    sector_dscr_floor: float
    meets_dscr_floor: bool
    leverage_ratio: float
    current_ratio: float
    revenue_variance_pct: float
    internal_grade: Literal["A1", "A2", "B1", "B2", "C1", "C2", "D1", "D2"]
    indicative_rate_pct: float
    reason_codes: str
    risk_band: Literal["approve", "borderline", "decline"]
    approval_authority: Literal["Credit Officer", "Board Credit Committee", "Credit Committee"]
    auto_approve_allowed: bool
    requires_human_signoff: bool

api_app = FastAPI(title="Underwriting Assistant", version="1.0")

@api_app.get("/health")
def health():
    return {"status": "ok"}   # cheap liveness check -- no LLM call, so it never hits a rate limit

@api_app.post("/chat", response_model=ChatResponse)
async def chat(req: ChatRequest):
    seed_report = {
        "raw": str(req.model_dump()),
        "proposed_annual_debt_service_inr": "",
        "dscr": "",
        "sector_dscr_floor": "",
        "meets_dscr_floor": "",
        "leverage_ratio": "",
        "current_ratio": "",
        "revenue_variance_pct": "",
        "internal_grade": "",
        "indicative_rate_pct": "",
        "reason_codes": "",
        "risk_band": "",
        "approval_authority": "",
        "auto_approve_allowed": False,
        "requires_human_signoff": True,
        "loan_application": {},
        "missing_fields": [],
        "log": [],
        "revision_count": 0,
        "critic_decision": "",
        "critic_feedback": "",
        "final_status": ""
        }

    #Invoke the workflow
    dcfg = {"configurable": {"thread_id": "committe-run"}}
    report = await multi_agent_graph.ainvoke(seed_report, dcfg)

    final_report = {key: report[key] for key in keys_to_extract}

    return ChatResponse(**final_report)

print("FastAPI app defined:", [r.path for r in api_app.routes if hasattr(r, "path")])

FastAPI app defined: ['/openapi.json', '/docs', '/docs/oauth2-redirect', '/redoc', '/health', '/chat']


In [33]:
import uvicorn, requests

_config = uvicorn.Config(api_app, host="0.0.0.0", port=8000, log_level="warning", reload=True)
_server = uvicorn.Server(_config)

def _run_server():
    # A dedicated event loop for this thread, driven with run_until_complete rather than
    # server.run() -- see the nest_asyncio note above.
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    loop.run_until_complete(_server.serve())

_server_thread = threading.Thread(target=_run_server, daemon=True)
_server_thread.start()
time.sleep(1.5)

# Real self-test -- an actual HTTP round trip, not a mocked response.
health = requests.get("http://127.0.0.1:8000/health", timeout=5)
print("Local health check:", health.status_code, health.json())

Local health check: 200 {'status': 'ok'}


In [ ]:
# manual check
# inp = {'application_id': 'ASH-L-00003', 'applicant_id': 'ASH-A-00003'}
# await chat(ChatRequest(**inp))

In [ ]:
# Real self-test on /chat -- shows the actual JSON payload a Postman call would get back.
resp = requests.post("http://127.0.0.1:8000/chat",
                      json={'application_id': 'ASH-L-00003', 'applicant_id': 'ASH-A-00003'}, timeout=150)

In [37]:
print(resp.status_code)
print(json.dumps(resp.json(), indent=2))

200
{
  "proposed_annual_debt_service_inr": 144039.0,
  "dscr": 0.38,
  "sector_dscr_floor": 1.25,
  "meets_dscr_floor": false,
  "leverage_ratio": 14.01,
  "current_ratio": 1.51,
  "revenue_variance_pct": 16.8,
  "internal_grade": "D2",
  "indicative_rate_pct": 18.5,
  "reason_codes": "DSCR_BELOW_FLOOR;PAST_DELINQUENCY;DOC_MISMATCH;LEVERAGE_ABOVE_LIMIT",
  "risk_band": "decline",
  "approval_authority": "Credit Officer",
  "auto_approve_allowed": false,
  "requires_human_signoff": true
}
